# Per-round Performance Score (PS_round) — beginner version

**Goal:** compute a per-round score for each fighter that reflects MMA judging priorities:

1) **Damage** (highest priority)  
2) **Dominance** (positional/skill supremacy that threatens damage)  
3) **Duration** (time share imposing effective offense that produces damage/dominance)

This PS_round is used as Tier A evidence:
- helps estimate how decisive a round/fight was,
- supports 10–9 vs 10–8 likelihood heuristics,
- improves rating updates when full round stats exist.

---

## 1) What we can measure (UFCStats proxies)

UFCStats does not measure “damage” directly, so we use proxies:

**Damage proxies**
- Knockdowns (KD)
- Significant strikes landed (weighted by target: head > body > leg)
- Sub attempts and threatening ground offense (proxy)

**Dominance proxies**
- Control time
- Takedowns and reversals
- Ground offense (but only if it creates threat)

**Duration proxies**
- Control share
- Pace share (attempt volume) as a rough “offensive time share”

---

## 2) The key rule: control without damage must be gated

Holding position alone should *not* produce a huge PS.
So we apply a gating function:

- First compute a **threat score** $T$ from damage proxies.
- Then multiply dominance/duration by a smooth gate $g(T)$.

If threat is low, dominance/duration contribute very little.

---

## 3) Outputs

For each round:
- `PS_round` in [0,1] (0.5 means roughly even)
- `dmg_index`, `dom_index`, `dur_index`
- optional `p10_9`, `p10_8`, `p10_7` heuristics

This notebook demonstrates a readable v1 version.


In [ ]:
import sys
from dataclasses import asdict
from importlib import import_module
from pathlib import Path

import pandas as pd

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

ps_module = import_module('elo_calculator.application.ranking.ps')
types_module = import_module('elo_calculator.application.ranking.types')
compute_ps_round = ps_module.compute_ps_round
FighterRoundStats = types_module.FighterRoundStats
RoundMeta = types_module.RoundMeta

## 4) Demo: control-heavy but low-damage vs damage-heavy rounds

We’ll create two toy rounds:

### Round 1: Fighter A has lots of control but little damage
- This should *not* look like a 10–8 unless threat is also high.

### Round 2: Fighter A lands more head sig strikes and a KD
- This should strongly increase the score.


In [ ]:
a_stats_round_1 = FighterRoundStats(
    kd=0,
    sig_landed=4,
    sig_attempted=12,
    td_landed=2,
    td_attempted=4,
    sub_attempts=0,
    rev=0,
    ctrl_seconds=180,
    head_landed=2,
    body_landed=1,
    leg_landed=1,
    ground_landed=1,
)

b_stats_round_1 = FighterRoundStats(
    kd=0,
    sig_landed=6,
    sig_attempted=14,
    td_landed=0,
    td_attempted=1,
    sub_attempts=0,
    rev=0,
    ctrl_seconds=20,
    head_landed=3,
    body_landed=2,
    leg_landed=1,
    ground_landed=0,
)

round_1_meta = RoundMeta(round_num=1)
round_1_result = compute_ps_round(a_stats_round_1, b_stats_round_1, round_1_meta)

pd.DataFrame([asdict(round_1_result)])

In [ ]:
a_stats_round_2 = FighterRoundStats(
    kd=1,
    sig_landed=14,
    sig_attempted=22,
    td_landed=1,
    td_attempted=2,
    sub_attempts=1,
    rev=0,
    ctrl_seconds=40,
    head_landed=10,
    body_landed=3,
    leg_landed=1,
    ground_landed=3,
)

b_stats_round_2 = FighterRoundStats(
    kd=0,
    sig_landed=8,
    sig_attempted=20,
    td_landed=0,
    td_attempted=1,
    sub_attempts=0,
    rev=0,
    ctrl_seconds=30,
    head_landed=4,
    body_landed=2,
    leg_landed=2,
    ground_landed=1,
)

round_2_meta = RoundMeta(round_num=2)
round_2_result = compute_ps_round(a_stats_round_2, b_stats_round_2, round_2_meta)

pd.DataFrame([asdict(round_2_result)])

## 5) What to look for in the outputs

- `gateA` should be relatively low in the “control without threat” scenario.
- In the damage-heavy scenario, threat increases and the gate opens, allowing dominance/duration to matter.
- `p10_8_A` / `p10_7_A` should remain small unless damage + dominance are both large.

This design is intentionally conservative: it tries to avoid false 10–8 signals.
